In [ ]:
import os
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Load environment variables
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize Pinecone
pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/intfloat/e5-small-v2"
    )


In [ ]:
from pinecone import ServerlessSpec

pc.create_index(
    name="my-index",
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [36]:
index = pc.Index("my-index")

In [25]:
Dir=r"C:\Users\4187v\Downloads\Medicine Project\Data"

In [37]:
def read_pdf(dir):
    loader = PyPDFDirectoryLoader(Dir)

    documents = loader.load()
    return documents

In [22]:
def chunk(docs, chunk_size=500,chunk_overlap=100):
    text_splitters=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    docs=text_splitters.split_documents(docs)
    return docs

In [ ]:
    doc=read_pdf(documents)
    documents=chunk(docs=doc)
    records = []

    for i, d in enumerate(documents):
        file_name = d.metadata["source"]
        file_name = os.path.basename(file_name)
        print(f"{file_name}_{i}")
        vector = embeddings.embed_query(d.page_content)
        records.append({
            "id": f"{file_name}_{i}",
            "values": vector,
            "metadata": {
                "text": d.page_content
            }
        })

    data_size=100
    for i in range(0, len(records), data_size):
        data = records[i:i+data_size]
        
        
        index.upsert(
            namespace="example-namespace",
            vectors=data
        )

In [ ]:
def CalSum(a,b):
    docsg=read_pdf(doc)
    documents=chunk(docs=docsg)

In [ ]:
#load_data()
def start_loading(Dir):
    for file in os.listdir(Dir):
        #if file.endswith(".docx"):
            path = os.path.join(Dir, file)
            load_data(path)
            print(f"File: {path}")

start_loading(Dir)